# 第3章 RAG 実践

この Notebook では、Pythonで根拠を検索し、その根拠をブラウザ Chat UI に渡して答えが変わることを体験します。さらに Agentic AI で、どの文書を根拠として読ませるかを体験します。

In [ ]:
from pathlib import Path
import os
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    configure_local_caches,
    copy_to_clipboard,
    load_chapter,
    ollama_generate,
    open_aider_terminal,
    prepare_aider_practice_workspace,
    print_headings,
    read_text,
    retrieve_chunks,
    split_markdown,
    REPO_ROOT,
    DOCS_DIR,
    WORK_DIR,
    start_ollama_chat_ui,
    stop_ollama_chat_ui,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("03-rag.md")
print(chapter_path)
print_headings(chapter_text)


## 1. Python で根拠を探す

まず `docs/local-llm-customization` のMarkdownを小さな単位に分け、質問に近い根拠を探します。

In [ ]:
chunks = []
for path in sorted(DOCS_DIR.glob("0[1-7]-*.md")):
    chunks.extend(split_markdown(path, read_text(path)))

question = "手順書を根拠にFAQを作りたい時、LoRAではなくRAGから始める理由は何ですか？"
evidence = retrieve_chunks(chunks, question, k=4)

print("質問:", question)
print("\n見つかった根拠:")
for i, item in enumerate(evidence, 1):
    print(f"[{i}] {item['source']} / {item['title']}")
    print(item['text'][:250].replace('\n', ' '), "\n")

## 2. Chat UI を開き、根拠なしで質問する

次のセルでブラウザ Chat UI を開き、質問をクリップボードへコピーします。開いた画面で貼り付けて送信し、どんな答えになるか見ます。

In [ ]:
start_ollama_chat_ui()
copy_to_clipboard(question)

## 3. 根拠を渡して、答えを比べる

次のセルは、検索した根拠と質問をまとめてクリップボードへコピーします。Chat UI に貼り付けて送信し、根拠なしの返答と比べます。

In [ ]:
context = "\n\n".join(f"[{i}] {item['source']} / {item['title']}\n{item['text']}" for i, item in enumerate(evidence, 1))
rag_text = f"""
次の根拠だけを使って質問に答えてください。
最後に参照した根拠番号を [1] のように示してください。

# 根拠
{context}

# 質問
{question}
""".strip()

copy_to_clipboard(rag_text)
print(rag_text[:1500])
print("\n=== Python API で根拠つき回答を再現 ===")
print(ollama_generate(rag_text, temperature=0.1))


## 4. Agentic AI で根拠ファイルを読ませる

次は aider を起動し、RAGの対象にしたい文書を明示的に追加します。開いた PowerShell で次を入力します。

1. `/help`
2. `/add docs/local-llm-customization/03-rag.md`
3. `/add docs/local-llm-customization/01-overview.md`
4. `追加した文書だけを根拠に、RAGを最初に試す理由を説明してください。まだ編集しないでください。`
5. `/exit`

ここで体験するのは、Agentic AI に「どのファイルを読ませるか」を人間が指定できることです。
実リポジトリを誤って編集しないように、この章では `work/aider-practice/` に作る練習用 workspace を開きます。ここは `.gitignore` されているため、教材の tracked files は変更されません。


In [ ]:
practice_path = prepare_aider_practice_workspace()
open_aider_terminal(practice_path)


## 結果の読み方 / 次へ進む判断

- 検索結果に質問と関係する章名や見出しが出れば、根拠候補を集める段階は成功です。
- 根拠なしの Chat UI 回答と、根拠を渡した回答で内容や参照番号が変われば、RAG の役割を体験できています。
- aider では `/add` した文書だけを根拠に説明させ、どのファイルを読ませるかを人間が選べることを確認します。

ここまで進めば、第4章で Chat UI、API、agentic coding、レビューを作業の入口として使い分けられます。

Chat UI を使い終わったら、同じ kernel で `stop_ollama_chat_ui()` を実行すると教材用サーバーを終了できます。Notebook の kernel を再起動しても終了します。
